In [ ]:
model_paths = {
#                 '/kaggle/input/pii-deberta-models/cuerpo-de-piiranha' : (3)/10,
#                 '/kaggle/input/pii-models/piidd-org-sakura' : (6)/10,
#                 '/kaggle/input/pii-detect-deberta3large-models/splendid-totem-15-checkpoint-1800-f1_0.9659' : (3)/10,
#                 '/kaggle/input/pii-deberta-models/cabeza-de-piiranha': (10/3)/10,
#                 '/kaggle/input/pii-deberta-models/cabeza-del-piinguuino': 9/10,
#                 '/kaggle/input/pii-deberta-models/cola del piinguuino': 1/10,
#                 '/kaggle/input/pii-detect-deberta3large-models/eager-water-37-fold1-checkpoint-650-f1_0.9661/': 10/10,
#                 '/kaggle/input/37vp4pjt': (10/3)/10,
#                 '/kaggle/input/pii-detect-deberta3large-models/pretty-darkness-46-chk-2350-f1_0.9539/': (1)/10
#                 '/kaggle/input/pii-detect-deberta3large-models/worthy-waterfall-chk-3200-f1_0.9590': 4/10
                '/kaggle/input/pii-deberta-models/cuerpo-de-piiranha' : (1/3)/10,
                '/kaggle/input/pii-models/piidd-org-sakura' : (1/3)/10,
                '/kaggle/input/pii-detect-deberta3large-models/splendid-totem-15-checkpoint-1800-f1_0.9659' : (1/3)/10,
                '/kaggle/input/pii-deberta-models/cabeza-del-piinguuino': 8/10,
                '/kaggle/input/37vp4pjt': (1)/10,
                }

threshold = 0.99

INFERENCE_MAX_LENGTH = 3500

# Load libraries

In [ ]:
import os
import re

import json
import argparse
from itertools import chain
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments, DataCollatorForTokenClassification
from datasets import Dataset
import numpy as np

# NLP Tokenization and Model Preparation

In [ ]:


# Function definition for 'tokenize'
def tokenize(example, tokenizer):
    # Initializing two lists: 'text' for storing tokens and 'token_map' for mapping tokens to their original positions.
    text = []
    token_map = []
    
    # Starting index at 0 to track tokens.
    idx = 0
    
    # Iterating through tokens and their associated trailing whitespaces.
    for t, ws in zip(example["tokens"], example["trailing_whitespace"]):
        
        # Adding each token to the 'text' list.
        text.append(t)
        
        # Extending 'token_map' with the current index repeated as many times as the length of the token.
        token_map.extend([idx] * len(t))
        
        # Adding a space for trailing whitespace and marking it with '-1' in 'token_map'.
        if ws:
            text.append(" ")
            token_map.append(-1)
            
        # Incrementing 'idx' for the next token.
        idx += 1
        
    # Tokenizing the concatenated 'text' and returning offset mappings along with 'token_map'.
    tokenized = tokenizer("".join(text), return_offsets_mapping=True, truncation=False, max_length=INFERENCE_MAX_LENGTH)
    
    # Returning a dictionary containing the tokenized data and the 'token_map'.
    return {
        **tokenized,
        "token_map": token_map,
    }


This code block outlines a comprehensive approach to processing and analyzing textual data, specifically aimed at detecting and removing Personally Identifiable Information (PII) using pretrained models. Here's a step-by-step explanation:

1. **Loading the Test Data**:
   - The code begins by loading a JSON file that contains the test data using `json.load()`. This data is expected to contain various fields relevant to the documents being analyzed.

2. **Dataset Creation**:
   - A dataset is created from the loaded data using `Dataset.from_dict()`, organizing the data into a structured format suitable for processing. This includes fields like `full_text`, `document`, `tokens`, and `trailing_whitespace` for each entry in the dataset.

3. **Tokenizer Initialization**:
   - The code initializes a tokenizer for the first model in a predefined list of model paths. This tokenizer is used to process the text data into a format that the models can understand.

4. **Tokenization and Parallel Processing**:
   - The dataset is then tokenized using the `map()` function, which applies a tokenization function across the dataset in parallel, enhancing processing efficiency.

5. **Model Path Definition and Weighting**:
   - A dictionary defines the paths to pretrained models and their respective weights, indicating the importance or contribution of each model to the final prediction.

6. **Intermediate Prediction Storage Setup**:
   - A directory for storing intermediate predictions is created, facilitating the aggregation of results from different models.

7. **Model Prediction and Weighting**:
   - For each model, the code loads the tokenizer and the model, predicts the token classifications for the dataset, and applies the predefined weight to these predictions. The predictions are then saved to the intermediate directory.

8. **Memory Management**:
   - After processing each model, the code performs garbage collection and clears the GPU cache to free up memory, ensuring efficient resource utilization.

9. **Aggregation of Predictions**:
   - The code aggregates the weighted predictions from all models by loading the intermediate prediction files and summing them up. This step combines the insights from different models into a comprehensive prediction.

10. **Final Prediction Calculation**:
    - Finally, the aggregated predictions are divided by the total weight to compute the weighted average of predictions across all models. This final step yields a consensus prediction that takes into account the contributions of each individual model based on their assigned weights.


In [ ]:
data = json.load(open("/kaggle/input/pii-detection-removal-from-educational-data/test.json"))

# Create a dataset from the loaded data
ds = Dataset.from_dict({
    "full_text": [x["full_text"] for x in data],
    "document": [x["document"] for x in data],
    "tokens": [x["tokens"] for x in data],
    "trailing_whitespace": [x["trailing_whitespace"] for x in data],
})

# Initialize a tokenizer and model from the pretrained model path
# model_paths = {'/kaggle/input/pii-deberta-models/cola-de-piiranha' : 2/10,
#               '/kaggle/input/pii-deberta-models/cuerpo-de-piiranha' : 2/10,
#               '/kaggle/input/pii-deberta-models/cabeza-de-piiranha' : 2/10,
#               '/kaggle/input/pii-deberta-models/cabeza-del-piinguuino' : 5/10}


first_model_path = list(model_paths.keys())[0]

tokenizer = AutoTokenizer.from_pretrained(first_model_path)

# Tokenize the dataset using the 'tokenize' function in parallel
ds = ds.map(tokenize, fn_kwargs={"tokenizer": tokenizer}, num_proc = 2)


#display(Image(filename='/kaggle/input/pii-deberta-models/piiratefisk.png'))

import gc
import torch
import numpy as np

from scipy.special import softmax


all_preds = []

# Calculate the total weight
total_weight = sum(model_paths.values())

# Directory for saving intermediate predictions
intermediate_dir = './intermediate_predictions'
os.makedirs(intermediate_dir, exist_ok=True)

for idx, (model_path, weight) in enumerate(model_paths.items()):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=16)
    args = TrainingArguments(
        ".",
        per_device_eval_batch_size=1,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collator,
        tokenizer=tokenizer,
    )
    predictions = trainer.predict(ds).predictions
    weighted_predictions = softmax(predictions, axis=-1) * weight
    # Save weighted_predictions to disk
    np.save(os.path.join(intermediate_dir, f'weighted_preds_{idx}.npy'), weighted_predictions)
    
    # Clear memory
    del model, trainer, tokenizer, predictions, weighted_predictions
    torch.cuda.empty_cache()
    gc.collect()

# Initialize an array for aggregated predictions
aggregated_predictions = None

# Load and aggregate predictions
for file_name in os.listdir(intermediate_dir):
    weighted_predictions = np.load(os.path.join(intermediate_dir, file_name))
    if aggregated_predictions is None:
        aggregated_predictions = weighted_predictions
    else:
        aggregated_predictions += weighted_predictions

# Finally, compute the weighted average of predictions
weighted_average_predictions = aggregated_predictions / total_weight


This code snippet is focused on processing the aggregated predictions to make final determinations about the classification of tokens in the text data, particularly in the context of detecting and removing Personally Identifiable Information (PII). Here's a detailed explanation of the steps involved:

1. **Configuration Loading**:
   - It begins by loading a configuration file (`config.json`) for the model, which is typically located within the model's directory. This configuration file contains important model parameters and metadata, such as the mapping from ID numbers to label names (`id2label`), which is critical for interpreting the model's predictions.

2. **Prediction Processing**:
   - The aggregated predictions (`weighted_average_predictions`) are processed to determine the most likely label for each token. This is done in two steps:
     - `preds` captures the index of the maximum value across the last dimension of the predictions array, representing the most likely label for each token considering all possible labels.
     - `preds_without_O` specifically focuses on non-'O' labels (where 'O' typically represents the 'Outside' or 'Other' category in token classification tasks), by excluding the 'O' label from consideration and finding the argmax across the modified label set. This helps in identifying tokens that have a significant likelihood of representing specific information without being overshadowed by the 'O' label's dominance.

3. **Threshold Application for 'O' Labels**:
   - The code defines a threshold value (`threshold = 0.99`) to decide whether to classify a token as 'O' (typically representing a token that does not correspond to any specific entity or information of interest) or as another label. This threshold is applied to the predictions for the 'O' label (`O_preds`), which are part of the `weighted_average_predictions`.
   - If the confidence score for the 'O' label is below the threshold, indicating that the model is less certain that the token should be classified as 'O', the prediction from `preds_without_O` is used. Otherwise, the original prediction (`preds`) is retained. This approach allows for a more nuanced handling of tokens that might be near the boundary of being classified as 'O' or as containing potentially identifiable information.

4. **Final Predictions**:
   - The result of this process is `preds_final`, an array of predictions that integrates both the specific interest in non-'O' labels and the careful consideration of the 'O' label based on a confidence threshold. This final prediction set is better suited for tasks where accurately identifying non-generic information is crucial, such as in PII detection and removal.

This method enhances the specificity and accuracy of PII detection by judiciously handling the balance between identifying non-'O' entities and avoiding false positives through the strategic use of a confidence threshold for 'O' labels.


In [ ]:
config = json.load(open(Path(model_path) / "config.json"))
id2label = config["id2label"]
preds = weighted_average_predictions.argmax(-1)
preds_without_O = weighted_average_predictions[:,:,:12].argmax(-1)
O_preds = weighted_average_predictions[:,:,12]



preds_final = np.where(O_preds < threshold, preds_without_O , preds)

This code snippet processes the final predictions to extract valuable information based on the predicted labels, focusing on tokens identified as containing or not containing PII (Personally Identifiable Information). The process aims to filter out irrelevant tokens and consolidate meaningful data into a structured format for analysis. Here’s a step-by-step explanation:

1. **Initialization**:
   - `triplets` and `processed` are initialized to store processed data, with `triplets` presumably intended for use (though not utilized within this snippet).
   - `pairs` is a set designed to efficiently track (document, token_id) pairs to avoid processing duplicates, leveraging the O(1) lookup time of sets.

2. **Data Processing Loop**:
   - The loop iterates over each prediction alongside related data (token mappings, offsets, tokens, documents) from the dataset.
   - For each token within a document, it examines the prediction (`p`), the corresponding start and end offsets of the token (`offsets`), and the token's actual text (`tokens`).

3. **Label and Token Processing**:
   - Each token's predicted label is determined by mapping the prediction (`token_pred`) to its label (`id2label`).
   - Tokens are filtered based on their offsets and mappings:
     - Tokens mapped to -1 or whose start and end offsets sum to zero are skipped, as these typically represent padding or irrelevant tokens.
     - Whitespace tokens or those not contributing to entity recognition (e.g., "\n\n") are also skipped.

4. **Entity Recognition**:
   - For tokens of interest (excluding "O", "B-EMAIL", "B-PHONE_NUM", "I-PHONE_NUM" labels), the script identifies unique tokens within documents. It does this by constructing a `pair` of the document ID and the token ID, adding meaningful tokens to the `processed` list if they haven't been processed before.
   - This approach ensures that each token identified as potentially containing PII is processed only once, avoiding redundancy.

5. **Output**:
   - The `processed` list accumulates structured data entries for each relevant token, including the document ID, token ID, predicted label, and the token's text. This structured format is ready for further analysis or reporting tasks, focusing on the detection and handling of PII within the dataset.

This methodical approach efficiently filters and organizes data from a set of predictions, streamlining the identification and extraction of potentially sensitive information within a large dataset.


In [ ]:
triplets = []
pairs = set()  # membership operation using set is faster O(1) than that of list O(n)

processed = []

# For each prediction, token mapping, offsets, tokens, and document in the dataset
for p, token_map, offsets, tokens, doc in zip(preds_final, ds["token_map"], ds["offset_mapping"], ds["tokens"], ds["document"]):

    # Iterate through each token prediction and its corresponding offsets
    for token_pred, (start_idx, end_idx) in zip(p, offsets):
        label_pred = id2label[str(token_pred)]  # Predicted label from token

        # If start and end indices sum to zero, continue to the next iteration
        if start_idx + end_idx == 0:
            continue

        # If the token mapping at the start index is -1, increment start index
        if token_map[start_idx] == -1:
            start_idx += 1

        # Ignore leading whitespace tokens ("\n\n")
        while start_idx < len(token_map) and tokens[token_map[start_idx]].isspace():
            start_idx += 1

        # If start index exceeds the length of token mapping, break the loop
        if start_idx >= len(token_map):
            break

        token_id = token_map[start_idx]  # Token ID at start index

        # Ignore "O" predictions and whitespace tokens
        if label_pred in ("O", "B-EMAIL", "B-PHONE_NUM", "I-PHONE_NUM") or token_id == -1:
            continue

        pair = (doc, token_id)

        if pair not in pairs:
            processed.append({"document": doc, "token": token_id, "label": label_pred, "token_str": tokens[token_id]})
            pairs.add(pair)
# We've gathered the valuable triplets from the dataset, ready for analysis!


This Python code snippet demonstrates a function `find_span` that identifies and returns the spans (as lists of token indices) of a target sequence of tokens (`target`) within a larger document (`document`). Both the target and the document are represented as lists of strings (tokens). The function uses the `spacy.lang.en.English` class to process English text, though its primary operation is independent of the specific language processing features provided by spaCy. Here’s a breakdown of how the function works:

1. **Initialization**:
   - The `nlp` variable is an instance of `English`, a language-specific processor in spaCy designed for English text. This processor is capable of basic language tasks such as tokenization.
   - `find_span` is a function that takes two parameters: `target`, a list of strings representing the sequence of tokens to find; and `document`, a list of strings representing the tokens of the entire document.

2. **Span Finding Logic**:
   - The function initializes `idx`, which tracks the current index within the `target` sequence being matched against the document.
   - `spans` is a list that will accumulate the found spans, where each span is a list of indices representing the start and end of a `target` sequence within the `document`.
   - A temporary list `span` is used to build up each individual span of indices as the function iterates through the document.

3. **Iteration and Matching**:
   - The function iterates over each token in the `document` using its index and value.
   - If the current document token does not match the corresponding target token (indicated by `idx`), the function resets `idx` and `span`, effectively restarting the search from the current position.
   - When a token matches the current target token, the token's index is added to `span`, and `idx` is incremented to check for the next token in the target sequence.
   - If the end of the target sequence is reached (`idx == len(target)`), the entire span is added to `spans`, and the search is reset to look for another occurrence of the target sequence.

4. **Return Value**:
   - Once the entire document has been scanned, the function returns `spans`, which contains all the spans of the target sequence found in the document. Each span is represented as a list of indices marking the beginning and end of the sequence in the document.

This function is particularly useful for tasks such as entity recognition, information extraction, or any scenario where identifying the occurrence and location of specific sequences of tokens within larger texts is necessary.


In [ ]:
from spacy.lang.en import English
nlp = English()

def find_span(target: list[str], document: list[str]) -> list[list[int]]:
    idx = 0
    spans = []
    span = []

    for i, token in enumerate(document):
        if token != target[idx]:
            idx = 0
            span = []
            continue
        span.append(i)
        idx += 1
        if idx == len(target):
            spans.append(span)
            span = []
            idx = 0
            continue
    
    return spans

This code snippet demonstrates how to load a JSON file containing test data and use regular expressions to find and categorize email addresses and phone numbers within the text. It leverages the Python `re` module for regular expression operations and the `spacy` library for tokenization to match sequences of tokens in the text. Here's a step-by-step breakdown of the code:

1. **Loading Data**:
   - The data is loaded from a JSON file, which is expected to contain educational data for PII (Personally Identifiable Information) detection and removal.

2. **Regular Expressions Setup**:
   - Two regular expressions are compiled:
     - `email_regex` targets typical email address patterns.
     - `phone_num_regex` looks for phone numbers in two specific formats (e.g., `(123)456-7890` and `123.456.7890`).

3. **Initialization**:
   - Lists `emails` and `phone_nums` are initialized to collect the discovered email addresses and phone numbers, respectively.

4. **Email Detection**:
   - For each item in the dataset, the code iterates through the tokens. If a token fully matches the `email_regex`, it is added to the `emails` list along with metadata like its document ID, token index, and label (`"B-EMAIL"`).

5. **Phone Number Detection**:
   - The full text of each item is scanned for matches against the `phone_num_regex`.
   - For each match, the matched string is tokenized using spaCy's tokenizer. The resulting tokens are then used to find their spans in the original token list of the document using the `find_span` function defined previously.
   - Each token within a matched span is added to the `phone_nums` list. Tokens at the beginning of a span are labeled with `"B-PHONE_NUM"`, and subsequent tokens in the span are labeled with `"I-PHONE_NUM"`, following the IOB (Inside, Outside, Beginning) tagging scheme used in named entity recognition.

6. **Data Structure**:
   - Both `emails` and `phone_nums` lists hold dictionaries for each identified email or phone number, including:
     - `document`: The ID or name of the document where the PII was found.
     - `token`: The index of the token within its document.
     - `label`: A label indicating the type of PII and its position within the PII sequence (e.g., `B-EMAIL`, `I-PHONE_NUM`).
     - `token_str`: The string representation of the token.

This approach effectively extracts and labels specific types of PII from text data, preparing it for further processing, such as anonymization or removal, to protect sensitive information.


In [ ]:
data = json.load(open("/kaggle/input/pii-detection-removal-from-educational-data/test.json"))

email_regex = re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+')
phone_num_regex = re.compile(r"(\(\d{3}\)\d{3}\-\d{4}\w*|\d{3}\.\d{3}\.\d{4})\s")
emails = []
phone_nums = []

for _data in data:
    # email
    for token_idx, token in enumerate(_data["tokens"]):
        if re.fullmatch(email_regex, token) is not None:
            emails.append(
                {"document": _data["document"], "token": token_idx, "label": "B-EMAIL", "token_str": token}
            )
    # phone number
    matches = phone_num_regex.findall(_data["full_text"])
    if not matches:
        continue
    for match in matches:
        target = [t.text for t in nlp.tokenizer(match)]
        matched_spans = find_span(target, _data["tokens"])
    for matched_span in matched_spans:
        for intermediate, token_idx in enumerate(matched_span):
            prefix = "I" if intermediate else "B"
            phone_nums.append(
                {"document": _data["document"], "token": token_idx, "label": f"{prefix}-PHONE_NUM", "token_str": _data["tokens"][token_idx]}
            )

In [ ]:
df = pd.DataFrame(processed + phone_nums + emails)

# Assign each row a unique 'row_id'
df["row_id"] = list(range(len(df)))

# Display a glimpse of the first 100 rows of your data
display(df.head(100))

# Cast your findings into a CSV file for further exploration
df[["row_id", "document", "token", "label"]].to_csv("submission.csv", index=False)

# May the winds of fortune guide ye to untold discoveries!
